# Rover on Rough Terrain — Planning Challenge

A point rover lives on a 2D arena. It must drive from **start** to **goal**.

You are given:
- `field` — a `(H, W, 3)` terrain map with 3 channels you can see:
  - **ch 0 — rocks**: hard obstacles, binary `0` (no rock) / `1` (rock); lethal, must avoid
  - **ch 1 — mud**: continuous 0..1
  - **ch 2 — slope**: continuous 0..1
- `demos` — a handful of **expert trajectories** (lists of `(x, y)` points).
  Each demo runs between its *own* start/goal pair (NOT necessarily yours), so
  no single demo is the answer — they're there to reveal the terrain cost.
- `start`, `goal` — the 2D points YOUR path must connect (`x = column`, `y = row`).

**Motion model:** the rover is a point in continuous `(x, y)` space; your path
is any polyline of waypoints (not restricted to a grid). The metric samples
along every segment, so sparse waypoints can't skip over a rock or costly cell.

Your job: a path that reaches the goal and is about as terrain-cheap as the
experts'. A pre-provided baseline cost lets you run a planner immediately, but
its weights are wrong — so plan under it and you'll cost too much. The demos
reveal what the experts actually avoid.

**Two TODOs:** (1) implement the planner, (2) learn the cost from the demos.
Partial progress is fine — talk through your choices as you go.

### Setup &mdash; clone data &amp; import the prebuilt helpers

In [ ]:
import os, sys
import numpy as np

# Clone the data repo when running on Colab (no-op if files already exist). The
# prebuilt helpers (show / path_cost / evaluate / cost_map_baseline) and the
# data ship together in this repo.
if not os.path.exists('field.npy'):
    import subprocess
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

sys.path.insert(0, os.getcwd())
from challenge_utils import (field, demos, start, goal, H, W,
                             ROCK, MUD, SLOPE, ROCK_PENALTY,
                             show, path_cost, evaluate, cost_map_baseline,
                             api_help)
print('field', field.shape, '| demos', len(demos),
      '| start', start, 'goal', goal)

### The API you have (do not edit — it's all imported from `challenge_utils`)

The plumbing — data loading, plotting, the official scorer, and the baseline
cost — is prebuilt and imported for you. You only write the two functions in the
TODO cells. Here's everything you can call:

- **Data:** `field` `(H,W,3)` with channels `ROCK`/`MUD`/`SLOPE` (= 0/1/2),
  `demos` (expert `(x,y)` paths), `start`/`goal` (`x=col, y=row`), `H`, `W`.
- **`show(paths=None, cost_map=None, title='')`** — plot terrain (or any cost
  map), the demos (gray), and your paths (lime).
- **`cost_map_baseline(field) -> (H,W)`** — a *deliberately wrong* runnable cost
  (`1 + 5*slope`, ignores mud); rocks at `ROCK_PENALTY`. Fixing it is TODO 2.
- **`path_cost(path, cost_map) -> float`** — integrate *your* cost map along a
  path (lower is better). Score rollouts inside your planner with this.
- **`evaluate(path, cost_map) -> bool`** — the official PASS/FAIL scorer. PASS
  needs: endpoints near start/goal, the path feasible under `cost_map`, in
  bounds, no rock under the hidden true cost, and terrain cost ≤ the expert bar.

The true terrain cost is hidden behind `path_cost`/`evaluate` (recovering it
from the demos is the point). Run the next cell for signatures + docstrings;
`challenge_utils.py` is in the repo if you want to read the source.

Run this to see the signatures + docstrings inline:

In [ ]:
api_help()
help(path_cost)
help(evaluate)

### Visualization &mdash; render the demos over the terrain

In [ ]:
show(title='true cost map + expert demos (gray)')

## TODO 1 &mdash; implement the planner (`plan`); details in the cell

In [ ]:
# ---- TODO 1: a sampling-based rollout planner (CPU/numpy first) -----------
# Plan a cheap start->goal path under a given (H,W) cost_map. Suggested scheme
# (you may do otherwise): hold a current T-waypoint trajectory; each iteration
# sample K noisy rollouts around it, score each with path_cost(p, cost_map),
# and step toward the cheap ones (softmax / take-best).
def plan(cost_map, K=256, T=60, iters=30, seed=0):
    '''Return an (T,2) path of (x,y) points from start to goal.'''
    rng = np.random.default_rng(seed)
    # TODO: replace this straight line (which ignores cost_map) with the
    # sampling rollout loop.
    return np.linspace(start, goal, T)

### Run the planner under the baseline cost &mdash; expect `FAIL` (wrong weights)

In [ ]:
cm_base = cost_map_baseline(field)
path = plan(cm_base)
show(path, cost_map=cm_base, title='baseline cost: planned path')
evaluate(path, cm_base)

## TODO 2 &mdash; learn the cost from demos (`cost_map_learned`); details in the cell

In [ ]:
# ---- TODO 2: learn the cost from the expert demos ------------------------
# The baseline charges for the wrong terrain. Use the demos to build a cost map
# that explains the experts' detours, then re-plan. Pick an approach and justify
# it: visitation (cells experts use are cheap), feature regression / IRL (fit
# weights over [mud, slope]), nearest-demo / warping, ...
#
# What "good" means here: your cost only has to RANK terrain like the experts do
# -- cheap where they drive, costly where they detour -- so the planner routes
# the same way. You do NOT need to recover the true cost numerically; getting the
# ordering right (and keeping rocks impassable) is enough. The pass bar has
# comfortable tolerance, so an expert-like path that avoids the costly terrain
# clears it -- don't over-fit chasing an exact cost match.
def cost_map_learned(field, demos):
    '''Return an (H,W) cost map inferred from demos. Keep rocks impassable.'''
    # TODO: use the demos. (Falls back to the wrong baseline for now.)
    return cost_map_baseline(field)

### Run the learned solution &mdash; goal: `PASS`

In [ ]:
cm = cost_map_learned(field, demos)
path = plan(cm)
show(path, cost_map=cm, title='learned: planned path over learned cost map')
show(path, title='learned: planned path over terrain')
evaluate(path, cm)

### Bonus / discussion — make it fast

Your planner rolls out `K` trajectories every iteration. On a real robot
we replan in a closed loop at, say, 50 Hz with `K` in the thousands.

- **Vectorize** the rollout: evaluate all `K` candidates without a Python loop
  (batched numpy, or move the cost lookup + scoring to `torch` on GPU).
- Be ready to discuss: memory layout of the `K × T × 2` buffer, how `K` trades
  off against horizon `T`, host↔device transfer / sync cost, and what your
  per-replan time budget buys you at 50 Hz.

In [ ]:
# ---- BONUS TODO: vectorized / GPU rollout --------------------------------
# Re-implement the per-iteration rollout scoring without a Python loop over K.
# numpy: index cost_map with a (K, T) array of flattened cell ids.
# torch: keep cost_map + samples on cuda; one gather + sum.
# (Optional — sketch it even if you don't finish.)